<p align="center">
    <img src="https://beyondeducation.es/wp-content/uploads/2024/02/CUNEF-UNI50.png" width="500">
</p>


<h1 align="center"><b>Initial Assignment - Machine Learning [P375]</b></h1>
<h2 align="center">M2026-27 - Máster of Ciencia de Datos e Inteligencia Artificial (MCDIA-3)</h2>

| Full name | CUNEF email |
|---|---|
| Melanie Álvarez [second surname] | [melanie.alvarez]@cunef.edu |
| María Victoria del Campo [second surname] | [victoria.delcampo]@cunef.edu |
| Alicia Plaza [second surname] | [alicia.plaza]@cunef.edu |

**GitHub repository:** [https://github.com/mvdelcampo/machine-learning-project]

# **1. Business Problem Description**

Antimicrobial resistance (AMR) is a global public health crisis, it reduces therapeutic
options and complicates clinical decision-making, especially in acute care
settings. Ciprofloxacin (CIP) is a widely used fluoroquinolone antibiotic for treating various bacterial infections, particularly certain urinary tract infections. According to the European Centre for Disease Prevention and Control (ECDC), **30% of invasive *Escherichia coli* isolates in Spain were resistant to fluoroquinolones in 2024**, compared with the EU/EEA average of 22.5%, highlighting the relevance of investigating resistance to this antibiotic class.


### **Shortcomings of the current solution**

The conventional method for determining ciprofloxacin resistance is the antibiogram (culture and antibiotic susceptibility testing), which typically takes 48–72 hours. During this period, clinicians may need to prescribe empirical treatment without knowing whether the bacterial isolate is resistant to ciprofloxacin, potentially leading to ineffective therapy and delayed appropriate treatment. 


### **Opportunity**

Machine Learning can predict ciprofloxacin resistance using available susceptibility results from other antibiotics and patient characteristics, supporting clinical decision-making while the complete antibiogram is pending.


### **Limitations of previous work** 

Wishal and Sahara, *Predicting Multi-Drug Resistance in Bacterial Isolates Through Performance Comparison and LIME-based Interpretation of Classification Models*, used the same synthetic dataset to predict multidrug resistance (MDR), comparing five Machine Learning models (LR, RF, AdaBoost, XGBoost, LightGBM) and achieving performance metrics close to 99%. However, they used antibiotic resistance results both to define MDR target and as predictors, creating a risk of **target leakage** and making the problem almost deterministic and of little real predictive value.

### **Research question and technical formulation**
> ***Can the resistance status of ciprofloxacin be estimated from the susceptibility profiles of other antibiotics, bacterial taxa, and patient clinical features?***

- **Learning paradigm:** supervised binary classification
- **Target (y):** `CIP`, with **R (Resistent) = 1** and **S (Susceptible) = 0** for this initial operational definition.
- **Candidate features (X):** results from the remaining **14 antibiotics**, plus `Age`, `Gender`, `Souches`, `Diabetes`, `Hypertension`, `Hospital_before`, `Infection_Freq`.
- **Leakage guard:** neither `CIP` nor any feature calculated from CIP (such as MDR labels that include CIP) may enter X.

# **2. Data Source**

The dataset is the **Multi-resistance antibiotic susceptibility** dataset attributed to A. Hosni on Kaggle. The source file is downloaded from `https://www.kaggle.com/datasets/adilimadeddinehosni/multi-resistance-antibiotic-susceptibility/data`.

***Advantages:*** The dataset contains 10,710 bacterial isolates, including susceptibility results for 15 antibiotics and relevant clinical characteristics. Its size and variety of features make it suitable for investigating cross-resistance patterns and developing a binary classification model to predict ciprofloxacin resistance. Additionally, it is publicly available, supporting reproducibility.

***Disadvantages:*** The dataset is synthetic and simulates data from only one healthcare facility and includes a limited number of bacterial species, reducing its generalizability to real clinical environments. It also contains missing values, inconsistent categories, and limited clinical information.

# **3. Data loading**

In [20]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("./data")

bacteria_dataset = pd.read_csv(DATA_PATH / "Bacteria_dataset_Multiresictance.csv")
print("All datasets loaded successfully!")

All datasets loaded successfully!


# **4. Exploratory Data Analysis**

## 4.1. Data description and inspection

In [21]:
bacteria_dataset.shape

(10710, 27)

In [22]:
bacteria_dataset.head()

,ID,Name,Email,Address,age/gender,Souches,Diabetes,Hypertension,Hospital_before,Infection_Freq,...,AN,Acide nalidixique,ofx,CIP,C,Co-trimoxazole,Furanes,colistine,Collection_Date,Notes
0,S290,Elizabeth Lawrence,elizabeth.lawrence@example.com,"6350 Robinson Loaf Apt. 447, Paulfurt, RI 30252",37/F,S290 Escherichia coli,No,No,No,0.0,...,S,S,S,S,R,S,S,S,5 Fev 2025,Patient had fever??
1,S291,Tina Sanders,tina.sanders@example.com,"78594 Galloway Port Suite 762, South Tanyatown...",29/F,S291 Morganella morganii,True,No,No,3.0,...,S,S,S,S,S,S,S,S,08/05/2021,تم العلاج
2,S292,Erin Cooke,erin.cooke@example.com,"76661 Isaiah Manors, North Benjamin, HI 85195",77/F,S292 Proteus mirabilis,True,No,No,3.0,...,S,S,S,R,R,S,S,S,01/05/2022,NADA
3,S293,Mrs. Sharon Watson,mrs..sharon.watson@example.com,"1670 Thompson Way Apt. 520, Andrewbury, IL 11534",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-05,PACIENTO ENFERMO
4,S294,Erika Stewart,erika.stewart@example.com,"76838 Patrick Green Suite 538, Torresmouth, ME...",13/F,S294 Escherichia coli,No,Yes,No,1.0,...,R,S,S,S,S,S,S,S,error,NADA


In [23]:
inspection = pd.DataFrame({
    "dtype": bacteria_dataset.dtypes.astype(str),
    "non_null": bacteria_dataset.notna().sum(),
    "missing_pct": (bacteria_dataset.isna().mean() * 100).round(2),
    "n_unique": bacteria_dataset.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False)

inspection

,dtype,non_null,missing_pct,n_unique
Collection_Date,str,10041,6.25,13
AMX/AMP,str,10052,6.14,8
GEN,str,10067,6.00,8
Souches,str,10070,5.98,9298
Co-trimoxazole,str,10070,5.98,8
Infection_Freq,str,10071,5.97,8
Hospital_before,str,10073,5.95,4
Furanes,str,10073,5.95,8
colistine,str,10074,5.94,8
Diabetes,str,10075,5.93,4


The `Name`, `Email`, and `Address` columns contain personally identifiable information (PII) that could reveal a patient's identity. To protect patient privacy and follow data protection principles, this information should not be included in the analysis. Since these variables are not relevant to predicting antibiotic resistance, they are removed from the dataset. Additionally, the `Collection_Date` column is also dropped because it is not considered relevant to the project's prediction objective and is not expected to provide useful information for predicting ciprofloxacin resistance.

In [24]:
bacteria_dataset.drop(columns=['Name', 'Email', 'Address', 'Collection_Date'], inplace=True)

/var/folders/w6/tt5g659n54zdxvlqlfd5dy3c0000gn/T/ipykernel_1223/3470330596.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  bacteria_dataset.drop(columns=['Name', 'Email', 'Address', 'Collection_Date'], inplace=True)


## 4.2. Duplicates

No duplicate records, repeated IDs, or missing ID values were found in the dataset. This confirms that all records have unique and valid identifiers, so no further investigation or cleaning is required regarding duplicates.

In [25]:
print("Duplicated full records:", bacteria_dataset.duplicated().sum())
print("Repeated ID values:", bacteria_dataset['ID'].duplicated().sum())
print("Rows with a null ID:", bacteria_dataset['ID'].isna().sum())

Duplicated full records: 0
Repeated ID values: 0
Rows with a null ID: 0


## 4.3. Data cleaning

### 4.3.1 Age and gender

The raw `age/gender` values may contain a number and F/M separated by `/`. We inspect exceptions, split the fields and convert age to numeric to enable distribution and outlier analyses. Values outside the expected pattern must be reviewed rather than silently treated as valid.

In [26]:
bacteria_dataset["age/gender"].unique()

<StringArray>
['37/F', '29/F', '77/F',    nan, '13/F', '57/M', '68/F', '18/F', '36/F',
 '49/M',
 ...
  '1/M', '67/M', '32/M', '23/M', '70/M', '37/M', '77/M', '10/M', '34/M',
 '66/M']
Length: 185, dtype: str

In [27]:
columna = bacteria_dataset["age/gender"].astype("string")

columna[
    ~columna.str.fullmatch(r"\d+\s*/\s*[FM]", case=False, na=False)
].unique()

<StringArray>
[<NA>, 'missing', '?']
Length: 3, dtype: string

We will now split the age/gender column into two separate columns, `Age` and `Gender`. First, it converts the values to strings and removes unnecessary spaces. Then, invalid values such as "missing", "nan", and "?" are replaced with missing values (`pd.NA`). Finally, it splits the column using / as the separator.

This makes the data easier to analyze and prepares both variables for further preprocessing and Machine Learning.

In [28]:
age_gender = bacteria_dataset["age/gender"].astype("string").str.strip()
age_gender = age_gender.mask(
    age_gender.str.lower().isin(["missing", "nan", "?"])
)

bacteria_dataset[["Age", "Gender"]] = age_gender.str.split("/", expand=True)

In [29]:
bacteria_dataset['Age'] = pd.to_numeric(bacteria_dataset['Age'], errors='coerce')

In [30]:
bacteria_dataset.head()

,ID,age/gender,Souches,Diabetes,Hypertension,Hospital_before,Infection_Freq,AMX/AMP,AMC,CZ,...,Acide nalidixique,ofx,CIP,C,Co-trimoxazole,Furanes,colistine,Notes,Age,Gender
0,S290,37/F,S290 Escherichia coli,No,No,No,0.0,R,R,R,...,S,S,S,R,S,S,S,Patient had fever??,37,F
1,S291,29/F,S291 Morganella morganii,True,No,No,3.0,S,R,S,...,S,S,S,S,S,S,S,تم العلاج,29,F
2,S292,77/F,S292 Proteus mirabilis,True,No,No,3.0,S,R,S,...,S,S,R,R,S,S,S,NADA,77,F
3,S293,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PACIENTO ENFERMO,<NA>,<NA>
4,S294,13/F,S294 Escherichia coli,No,Yes,No,1.0,S,S,R,...,S,S,S,S,S,S,S,NADA,13,F


### 4.3.2. Clinical binary variables

The `Diabetes`, `Hypertension`, and `Hospital_before` columns are converted into binary variables (0 = No, 1 = Yes) to make them easier to analyze and use in Machine Learning models. Missing values are preserved for further processing.

In [31]:
mapping = {"true": 1, "yes": 1, "false": 0, "no": 0}

for col in ["Diabetes", "Hypertension", "Hospital_before"]:
    values = bacteria_dataset[col].astype("string").str.strip().str.lower()
    values = values.replace({"?": pd.NA, "missing": pd.NA, "": pd.NA})

    unexpected = set(values.dropna().unique()) - set(mapping)
    if unexpected:
        raise ValueError(f"Valores inesperados en {col}: {unexpected}")

    bacteria_dataset[col] = values.map(mapping).astype("Int64")

### 4.3.3. Bacterial species (`Souches`)

The `Souches` column contains a patient identification prefix (e.g., `S687`) followed by the bacterial species name. Some species names also contain spelling errors or inconsistent abbreviations. First, we identify these inconsistencies, then remove the identification prefixes and correct the names to ensure that each bacterial species is represented consistently. Invalid or missing values are also standardized as `pd.NA`.

In [32]:
# Remove the first word, preserving single-word values
souches = bacteria_dataset["Souches"].astype("string").str.strip()

souches_sin_id = (
    souches
    .str.split(n=1)
    .str[1]
    .fillna(souches)
    .str.strip()
)

# Check unique values
print("=== UNIQUE BACTERIAL SPECIES ===")
print(souches_sin_id.unique())

print("\n=== BACTERIAL SPECIES FREQUENCY ===")
print(souches_sin_id.value_counts(dropna=False))

print("\n=== TOTAL UNIQUE SPECIES ===")
print(souches_sin_id.nunique())

=== UNIQUE BACTERIAL SPECIES ===
['Escherichia coli' 'Morganella morganii' 'Proteus mirabilis' <NA>
 'Citrobacter spp.' 'Enterobacteria spp.' 'Proeus mirabilis'
 'Pseudomonas aeruginosa' 'Klebsiella pneumoniae' 'E. coli'
 'Klbsiella pneumoniae' 'E.cli' 'E.coi' 'Serratia marcescens'
 'Klebsie.lla pneumoniae' 'Acinetobacter baumannii' 'Prot.eus mirabilis'
 'missing' 'Protus mirabilis' 'Enter.bacteria spp.' '?'
 'Enteobacteria spp.']

=== BACTERIAL SPECIES FREQUENCY ===
Souches
Escherichia coli           4864
Enterobacteria spp.         802
<NA>                        640
Proteus mirabilis           598
Klebsiella pneumoniae       565
Citrobacter spp.            481
E.coi                       432
E.cli                       396
E. coli                     391
Morganella morganii         305
Serratia marcescens         256
Pseudomonas aeruginosa      200
Acinetobacter baumannii     181
Enter.bacteria spp.         100
Enteobacteria spp.           95
Klbsiella pneumoniae         74
Klebsie.

In [33]:
# Dictionary to correct bacterial species names
corrections = {
    'E. coli': 'Escherichia coli',
    'E.cli': 'Escherichia coli',
    'E.coi': 'Escherichia coli',

    'Klbsiella pneumoniae': 'Klebsiella pneumoniae',
    'Klebsie.lla pneumoniae': 'Klebsiella pneumoniae',

    'Proeus mirabilis': 'Proteus mirabilis',
    'Prot.eus mirabilis': 'Proteus mirabilis',
    'Protus mirabilis': 'Proteus mirabilis',

    'Enter.bacteria spp.': 'Enterobacteria spp.',
    'Enteobacteria spp.': 'Enterobacteria spp.'
}

# Clean and standardize bacterial species
bacteria_dataset["Souches"] = (
    bacteria_dataset["Souches"]
    .astype("string")
    .str.strip()
    .str.replace(r"^S\d+\s+", "", regex=True)  # Remove sample ID
    .str.strip()
    .replace(corrections)  # Correct typos
    .replace(
        r"(?i)^(missing|na|nan|none|null|\?)$",
        pd.NA,
        regex=True
    )  # Replace missing indicators with pd.NA
)

In [39]:
print("=== UNIQUE BACTERIAL SPECIES (AFTER CLEANING) ===")
print(bacteria_dataset["Souches"].unique())

print("\n=== BACTERIAL SPECIES FREQUENCY (AFTER CLEANING) ===")
print(bacteria_dataset["Souches"].value_counts(dropna=False))

print("\n=== TOTAL UNIQUE SPECIES (AFTER CLEANING) ===")
print(bacteria_dataset["Souches"].nunique())

=== UNIQUE BACTERIAL SPECIES (AFTER CLEANING) ===
<StringArray>
[       'Escherichia coli',     'Morganella morganii',
       'Proteus mirabilis',                      <NA>,
        'Citrobacter spp.',     'Enterobacteria spp.',
  'Pseudomonas aeruginosa',   'Klebsiella pneumoniae',
     'Serratia marcescens', 'Acinetobacter baumannii']
Length: 10, dtype: string

=== BACTERIAL SPECIES FREQUENCY (AFTER CLEANING) ===
Souches
Escherichia coli           6083
Enterobacteria spp.         997
<NA>                        763
Proteus mirabilis           742
Klebsiella pneumoniae       702
Citrobacter spp.            481
Morganella morganii         305
Serratia marcescens         256
Pseudomonas aeruginosa      200
Acinetobacter baumannii     181
Name: count, dtype: Int64

=== TOTAL UNIQUE SPECIES (AFTER CLEANING) ===
9


### 4.3.4. Antibiotic susceptibility results

The dataset contains susceptibility results for 15 antibiotics. In this step, we clean and standardize the values into three categories: `R` (Resistant), `S` (Susceptible), and `I` (Intermediate). We also identify invalid entries and convert them into missing values. Keeping these categories separate preserves the original susceptibility information for further analysis and preprocessing.

In [36]:
bacteria_dataset["AMC"].value_counts(dropna=False)

AMC
R               5719
S               3876
NaN              632
i                 97
r                 96
s                 94
Intermediate      75
missing           66
?                 55
Name: count, dtype: int64

In [37]:
antibioticos = [
    "AMX/AMP", "AMC", "CZ", "FOX", "CTX/CRO", "IPM", "GEN", "AN",
    "Acide nalidixique", "ofx", "CIP", "C", "Co-trimoxazole",
    "Furanes", "colistine"
]

# Normalizar texto: quitar espacios y unificar mayúsculas
valores = bacteria_dataset[antibioticos].astype("string")
valores = valores.apply(lambda columna: columna.str.strip().str.upper())

# Estandarizar marcadores de ausencia y "Intermediate"
valores = valores.replace({
    "": pd.NA,
    "MISSING": pd.NA,
    "?": pd.NA,
    "NAN": pd.NA,
    "INTERMEDIATE": "I"
})

# Comprobar que no queden categorías distintas de R, S, I o ausente
inesperados = {
    columna: sorted(valores[columna].dropna().unique().tolist())
    for columna in antibioticos
    if not valores[columna].dropna().isin(["R", "S", "I"]).all()
}

if inesperados:
    raise ValueError(f"Valores inesperados en columnas de antibióticos: {inesperados}")

bacteria_dataset[antibioticos] = valores

In [40]:
# Confirm vocabulary of all 15 antibiotic columns after standardization.
antibiotic_check = pd.DataFrame({
    "R": (bacteria_dataset[antibioticos] == "R").sum(),
    "S": (bacteria_dataset[antibioticos] == "S").sum(),
    "I": (bacteria_dataset[antibioticos] == "I").sum(),
    "missing": bacteria_dataset[antibioticos].isna().sum()
})

# Total count for each antibiotic
antibiotic_check["Total"] = antibiotic_check.sum(axis=1)

antibiotic_check

,R,S,I,missing,Total
AMX/AMP,5712,4061,184,753,10710
AMC,5815,3970,172,753,10710
CZ,5690,4097,170,753,10710
FOX,5735,4049,173,753,10710
CTX/CRO,5748,4026,183,753,10710
IPM,5717,4054,186,753,10710
GEN,1936,7854,167,753,10710
AN,1905,7875,177,753,10710
Acide nalidixique,1388,8379,190,753,10710
ofx,1381,8406,170,753,10710


### 4.3.5. Infection frequency and notes

After reviewing the `Notes` column, we found that it does not provide relevant information about the patients or useful features for predicting ciprofloxacin resistance. Therefore, this column is removed from the dataset to reduce unnecessary information.

In [43]:
for col in ["Infection_Freq", "Notes"]:
    print(f"\n=== {col} ===")

    print("Missing values:", bacteria_dataset[col].isna().sum())
    print("Unique values:", bacteria_dataset[col].nunique())
    display(bacteria_dataset[col].value_counts(dropna=False))


=== Infection_Freq ===
Missing values: 639
Unique values: 8


Infection_Freq
2.0        2966
1.0        2889
3.0        1963
0.0        1896
NaN         639
missing     131
unknown      81
error        81
?            64
Name: count, dtype: int64


=== Notes ===
Missing values: 112
Unique values: 9


Notes
see file               1508
MALADIE GRAVE          1507
Patient had fever??    1505
--                     1504
تم العلاج              1500
PACIENTO ENFERMO       1490
NADA                   1452
NaN                     112
?                        72
missing                  60
Name: count, dtype: int64

In [44]:
bacteria_dataset.drop(columns=['Notes'], inplace=True)

/var/folders/w6/tt5g659n54zdxvlqlfd5dy3c0000gn/T/ipykernel_1223/2953118997.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  bacteria_dataset.drop(columns=['Notes'], inplace=True)


## 4.4. Missing data analysis

## 4.5. Distributions and descriptive analysis

## 4.6. Target formulation: ciprofloxacin (CIP)

## 4.7. Outlier

# **5. Conclusions**